# `esmDMS` walkthrough: TpoR cellular and BF520 viral data

This notebook walks from raw data processing to ESM embedding and feature inference for two datasets in `data/raw_data`:

- **TpoR**: cellular/MaveDB-style nucleotide count data.
- **BF520**: viral pre/post codon-count data.

The workflow is intentionally class-first: instantiate the input/config objects, create an `esmDMS` object, call `process_raw_data()`, `embed_all_sequences()`, `run_feature_inference()`, and then use the plotting methods that load inference results by layer and metadata.


In [ ]:
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
# Import local popDMS first so esmDMS resolves the repo-local module even after
# embedding_scripts.embed_sequences adjusts sys.path during import.
#import popDMS
from esmDMS import CellularDMSInput, ESMDMSConfig, ViralDMSInput, esmDMS

RAW_DIR = Path("data/raw_data")
CACHE_DIR = Path("data/sequence_data/esmdms_walkthrough")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ESM_LAYER = "0"
PLOT_LAYERS = [ESM_LAYER]
# If you run inference for more layers, set for example: PLOT_LAYERS = [str(i) for i in range(6)]

config = ESMDMSConfig(
    embedding_model="esm2_t6_8M_UR50D",
    embedding_method="mean_pool",
    local_or_disk="both",
    save_dir=str(CACHE_DIR),
)

tpor_config = config.copy()
bf520_config = config.copy()
bf520_config.local_or_disk = "disk"


/net/dali/home/barton/dhw28/popDMS/esmDMS/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## TpoR cellular walkthrough

The cellular input uses a nucleotide reference sequence plus a MaveDB-style nucleotide count CSV.


In [2]:
tpor = esmDMS(
    CellularDMSInput(
        reference_nuc_path=RAW_DIR / "TpoR_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "TpoR_nucleotide_counts.csv",
    ),
    config=config,
)

tpor.process_raw_data(drop_stop_codons=True)

print(tpor.sequence_dataframe.shape)
tpor.sequence_dataframe.head()


(12948, 4)


,SequenceIndex,Replicate,Generation,Frequency
0,0,1,0,121
1,0,1,1,247
2,0,2,0,121
3,0,2,1,129
4,0,3,0,121


In [3]:
tpor.embed_all_sequences(layer=ESM_LAYER)
tpor_result = tpor.run_feature_inference(
    layer=ESM_LAYER,
    abstraction_method="Embeddings",
    abstraction_params={"norm_scheme": "per_feature"},
)

print("gamma_opt:", tpor_result.gamma_opt)
print("per-replicate s shape:", tpor_result.s.shape)
print("joint s shape:", tpor_result.s_joint.shape)


Loading weights: 100%|██████████| 107/107 [00:00<00:00, 359.42it/s, Materializing param=encoder.layer.5.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


No saved inference results found at data/sequence_data/esmdms_walkthrough/Embeddings_Layer_0_per_feature_inference_results.pkl. Running inference and saving results.
No abstraction method specified. Using raw embeddings as features.
Embeddings already exist in memory. Returning existing embeddings.
gamma_opt: 6.951927961775605
per-replicate s shape: (6, 320)
joint s shape: (320,)


## Replicate comparison plots

These cells load saved or in-memory inference results by layer/method/normalization and use the plotting methods on the `esmDMS` object. The layer-summary calls use `PLOT_LAYERS`; with the default quick run this is just `ESM_LAYER`, but the same code works for a larger list after running inference for those layers.


In [ ]:
tpor.plot_rep_sel_comps(
    layer=ESM_LAYER,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    label="TpoR",
)

tpor.plot_rep_fit_comps(
    layer=ESM_LAYER,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    label="TpoR",
)

tpor_sel_corr_fig, tpor_sel_corr = tpor.plot_avg_rep_correlations_by_layer(
    layers=PLOT_LAYERS,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    comparison="selection",
    label="TpoR",
)

tpor_fit_corr_fig, tpor_fit_corr = tpor.plot_avg_rep_correlations_by_layer(
    layers=PLOT_LAYERS,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    comparison="fitness",
    label="TpoR",
)

tpor_sel_corr, tpor_fit_corr


## BF520 viral walkthrough

The viral input uses paired pre-selection mutant-DNA codon counts and post-selection mutant-virus codon counts.


In [4]:
bf520 = esmDMS(
    ViralDMSInput(
        pre_files=(
            RAW_DIR / "BF520_mutDNA-1_codoncounts.csv",
            RAW_DIR / "BF520_mutDNA-2_codoncounts.csv",
            RAW_DIR / "BF520_mutDNA-3_codoncounts.csv",
        ),
        post_files=(
            RAW_DIR / "BF520_mutvirus-1_codoncounts.csv",
            RAW_DIR / "BF520_mutvirus-2_codoncounts.csv",
            RAW_DIR / "BF520_mutvirus-3_codoncounts.csv",
        ),
    ),
    config=config,
)

bf520.process_raw_data(drop_stop_codons=True)

print(bf520.sequence_dataframe.shape)
bf520.sequence_dataframe.head()


(75468, 4)


,SequenceIndex,Replicate,Generation,Frequency
0,0,1,0,0
1,0,1,1,0
2,0,2,0,0
3,0,2,1,0
4,0,3,0,0


In [3]:
bf520.embed_all_sequences(layer=ESM_LAYER)
bf520_result = bf520.run_feature_inference(
    layer=ESM_LAYER,
    abstraction_method="Embeddings",
    abstraction_params={"norm_scheme": "per_feature"},
)

print("gamma_opt:", bf520_result.gamma_opt)
print("per-replicate s shape:", bf520_result.s.shape)
print("joint s shape:", bf520_result.s_joint.shape)


Loading weights: 100%|██████████| 107/107 [00:00<00:00, 322.13it/s, Materializing param=encoder.layer.5.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KeyboardInterrupt: 

## BF520 replicate comparison plots

The same selection, fitness, and layer-summary plots can be generated from the BF520 inference results.


In [ ]:
bf520.plot_rep_sel_comps(
    layer=ESM_LAYER,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    label="BF520",
)

bf520.plot_rep_fit_comps(
    layer=ESM_LAYER,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    label="BF520",
)

bf520_sel_corr_fig, bf520_sel_corr = bf520.plot_avg_rep_correlations_by_layer(
    layers=PLOT_LAYERS,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    comparison="selection",
    label="BF520",
)

bf520_fit_corr_fig, bf520_fit_corr = bf520.plot_avg_rep_correlations_by_layer(
    layers=PLOT_LAYERS,
    abstraction_method="Embeddings",
    norm_scheme="per_feature",
    comparison="fitness",
    label="BF520",
)

bf520_sel_corr, bf520_fit_corr


## Cluster batch embedding example

For full datasets, use the cluster batch helpers instead of embedding every sequence in the notebook process. The calls below write Slurm array scripts and payload files, but do not submit jobs unless `submit=True`. After the array jobs finish, run `merge_embedding_batch_outputs(...)` to collect chunk outputs and populate the normal layer embedding cache used by inference.


In [5]:
TPOR_BATCH_DIR = CACHE_DIR / "tpor_embedding_batch"
BF520_BATCH_DIR = CACHE_DIR / "bf520_embedding_batch"

tpor_batch = tpor.create_embedding_batch_job(
    job_dir=TPOR_BATCH_DIR,
    n_chunks=2,
    job_name="tpor_esm_embed",
    partition="dept_cpu",
    mem="16G",
    time="01:00:00",
    scratch_root="/scr",
    submit=False,
)

bf520_batch = bf520.create_embedding_batch_job(
    job_dir=BF520_BATCH_DIR,
    n_chunks=3,
    job_name="bf520_esm_embed",
    partition="dept_cpu",
    mem="16G",
    time="01:00:00",
    scratch_root="/scr",
    submit=True,
)

print("TpoR script:", tpor_batch["script_path"])
print("BF520 script:", bf520_batch["script_path"])


TpoR script: data/sequence_data/esmdms_walkthrough/tpor_embedding_batch/submit_embedding_array.sh
BF520 script: data/sequence_data/esmdms_walkthrough/bf520_embedding_batch/submit_embedding_array.sh


After the Slurm array jobs complete, merge the returned chunk files. These lines are intentionally commented so the notebook can run before the cluster jobs have produced outputs.


In [ ]:
tpor.merge_embedding_batch_outputs(job_dir=TPOR_BATCH_DIR, layer=ESM_LAYER)
bf520.merge_embedding_batch_outputs(job_dir=BF520_BATCH_DIR, layer=ESM_LAYER)
#
# Once merged, run inference as above:
tpor_result = tpor.run_feature_inference(ESM_LAYER, "Embeddings", {"norm_scheme": "per_feature"})
bf520_result = bf520.run_feature_inference(ESM_LAYER, "Embeddings", {"norm_scheme": "per_feature"})


## Compare outputs

At this point both datasets have gone through the same class-level pipeline: raw data processing, ESM embedding for one layer, and popDMS inference on the resulting feature vectors.


In [ ]:
summary = {
    "TpoR": {
        "rows": len(tpor.sequence_dataframe),
        "sequences": len(tpor.sequence_to_protein_sequence),
        "replicates": tpor.sequence_dataframe["Replicate"].nunique(),
        "feature_dims": tpor_result.s_joint.shape[0],
    },
    "BF520": {
        "rows": len(bf520.sequence_dataframe),
        "sequences": len(bf520.sequence_to_protein_sequence),
        "replicates": bf520.sequence_dataframe["Replicate"].nunique(),
        "feature_dims": bf520_result.s_joint.shape[0],
    },
}
summary
